# Bias and fairness - part 2: mitigation, held out

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/bias-fairness/notebooks/ex4_mitigation.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Bias and fairness*

Post-processing: leave the model as it is and move its decision threshold,
separately for women and men, until a fairness constraint holds. The
thresholds are fitted on a **calibration** part and judged on a **test** part
they never saw.

**Three holes.** The unmitigated baseline, the two mitigations, and the
same mitigations drawn two hundred times - and, in the questions, what each
one cost.

# Data, split and model

The metric functions of part 1 are given here ready-made. The XGBoost
returns its probabilities as float64: `fairlearn`'s ThresholdOptimizer fails
on XGBoost's float32.

In [ ]:
%pip install -q fairlearn

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/tomasvicar/AUI-public/master/labs/robustness/data/heart_disease_cleveland.csv"
CSV = Path("heart_disease_cleveland.csv")
if not CSV.is_file():
    urllib.request.urlretrieve(URL, CSV)

SEED = 42
N_SPLITS = 20
N_DRAWS = 200
GROUPS = {0: "female", 1: "male"}


def load_heart(path: Path = CSV) -> tuple[pd.DataFrame, pd.Series]:
    """13 features (missing values filled with the median) and disease = num > 0."""
    table = pd.read_csv(path)
    y = (table.pop("num") > 0).astype(int)
    return table.fillna(table.median(numeric_only=True)), y


class XGBClassifier64(xgb.XGBClassifier):
    """XGBoost whose predict_proba returns float64: `fairlearn`'s
    ThresholdOptimizer writes its interpolated probabilities into an array of
    the model's output dtype and fails on XGBoost's float32. The numbers are
    the same."""

    def predict_proba(self, X, *args, **kwargs):
        return super().predict_proba(X, *args, **kwargs).astype(np.float64)


def train_xgboost(X_train: pd.DataFrame, y_train: pd.Series, seed: int = SEED):
    """The shared recipe of the tabular running example."""
    model = XGBClassifier64(n_estimators=200, max_depth=3, learning_rate=0.1,
                            eval_metric="logloss", random_state=seed)
    return model.fit(X_train, y_train)


def wilson(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """The Wilson score interval of a proportion (95 % by default)."""
    p = successes / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


X, y = load_heart()
for code, name in GROUPS.items():
    print(f"{name}: {(X['sex'] == code).sum()} patients, {y[X['sex'] == code].mean():.1%} with the disease")

In [ ]:
def group_metrics(y_true, y_pred, sex) -> pd.DataFrame:
    """Per group: size, number of sick patients, base rate, selection rate,
    TPR, FPR and PPV of the thresholded predictions."""
    y_true, y_pred, sex = np.asarray(y_true), np.asarray(y_pred), np.asarray(sex)
    rows = {}
    for code, name in GROUPS.items():
        g = sex == code
        t, p = y_true[g], y_pred[g]
        rows[name] = {
            "n": int(g.sum()),
            "sick": int(t.sum()),
            "base rate": t.mean(),
            "selection rate": p.mean(),
            "TPR": p[t == 1].mean() if (t == 1).any() else np.nan,
            "FPR": p[t == 0].mean() if (t == 0).any() else np.nan,
            "PPV": t[p == 1].mean() if (p == 1).any() else np.nan,
        }
    return pd.DataFrame(rows).T


def gaps(metrics: pd.DataFrame) -> dict:
    """Male minus female, for the four rates; the demographic-parity gap is
    the selection-rate difference, the equalized-odds gap the larger of the
    TPR and FPR differences (in absolute value)."""
    d = metrics.loc["male"] - metrics.loc["female"]
    return {"selection rate": d["selection rate"], "TPR": d["TPR"], "FPR": d["FPR"],
            "PPV": d["PPV"], "demographic parity": abs(d["selection rate"]),
            "equalized odds": max(abs(d["TPR"]), abs(d["FPR"]))}


# Train / calibration / test = 50 / 25 / 25 %: the thresholds are fitted on
# the calibration part, and the test part is touched once, at the end.
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.5,
                                                    random_state=SEED, stratify=y)
X_calib, X_test, y_calib, y_test = train_test_split(X_rest, y_rest, test_size=0.5,
                                                    random_state=SEED, stratify=y_rest)
model = train_xgboost(X_train, y_train)
for name, part, target in (("calibration", X_calib, y_calib), ("test", X_test, y_test)):
    women = part["sex"] == 0
    print(f"{name}: {len(part)} patients, {women.sum()} women ({target[women].sum()} sick)")

# The model as it is

### Hole 1 - the baseline on the test part

In [ ]:
# TODO 1: the unmitigated model on the TEST part - its predictions, their
#         accuracy, the group_metrics and the gaps. Keep the results in
#         `results["unmitigated"] = {"accuracy": ..., "metrics": ..., "gaps": ...}`.

results = {}
...

# Two mitigations

*Equalized odds* asks for equal TPR and FPR; *demographic parity* for equal
selection rates.

### Hole 2 - ThresholdOptimizer, twice

In [ ]:
# TODO 2: post-processing with fairlearn's ThresholdOptimizer:
#
#   ThresholdOptimizer(estimator=model, constraints=..., predict_method="predict_proba",
#                      prefit=True).fit(X_calib, y_calib, sensitive_features=X_calib["sex"])
#
#   then .predict(X_test, sensitive_features=X_test["sex"], random_state=SEED).
#   Do it for constraints="equalized_odds" and for "demographic_parity", and
#   store each as results[constraint] in the same form as in hole 1.
from fairlearn.postprocessing import ThresholdOptimizer

for constraint in ("equalized_odds", "demographic_parity"):
    ...

In [ ]:
summary = pd.DataFrame({
    name: {"accuracy": r["accuracy"],
           "TPR women": r["metrics"].loc["female", "TPR"], "TPR men": r["metrics"].loc["male", "TPR"],
           "FPR women": r["metrics"].loc["female", "FPR"], "FPR men": r["metrics"].loc["male", "FPR"],
           "demographic-parity gap": r["gaps"]["demographic parity"],
           "equalized-odds gap": r["gaps"]["equalized odds"]}
    for name, r in results.items()})
summary.round(3)

# Another draw

Between its two thresholds the ThresholdOptimizer decides by a weighted coin
(the lecture's slide *A coin flip in the decision*). Everything above is one
draw, `random_state=SEED`. Without `random_state` every run of the cell gives
another one.

### Hole 3 - two hundred draws

In [ ]:
# TODO 3: the ThresholdOptimizer flags a patient between its two thresholds
#         by a coin flip, so every predict() is one draw. For each constraint
#         fit the optimizer as in hole 2, then predict the TEST part with
#         random_state = 0 ... N_DRAWS - 1. Collect one row per constraint,
#         draw and group in a DataFrame `draws` with the columns constraint,
#         draw, group, accuracy, "equalized-odds gap", "demographic-parity gap"
#         and the metrics of group_metrics.

rows = []
for constraint in ("equalized_odds", "demographic_parity"):
    ...
draws = pd.DataFrame(rows)

In [ ]:
for constraint in ("equalized_odds", "demographic_parity"):
    c = draws[draws["constraint"] == constraint]
    women = c[c["group"] == "female"]
    tpr = c.groupby("group")["TPR"]
    print(f"{constraint}: TPR women / men - draw {SEED} "
          f"{c[(c.draw == SEED) & (c.group == 'female')]['TPR'].iloc[0]:.3f} / "
          f"{c[(c.draw == SEED) & (c.group == 'male')]['TPR'].iloc[0]:.3f}, "
          f"mean {tpr.mean()['female']:.3f} / {tpr.mean()['male']:.3f}; "
          f"equalized-odds gap: draw {SEED} {women[women.draw == SEED]['equalized-odds gap'].iloc[0]:.3f}, "
          f"mean {women['equalized-odds gap'].mean():.3f}, "
          f"range {women['equalized-odds gap'].min():.3f}-{women['equalized-odds gap'].max():.3f}; "
          f"accuracy mean {women['accuracy'].mean():.3f}")

eo = draws[draws["constraint"] == "equalized_odds"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for group, name, color in (("female", "women", "#2a78d6"), ("male", "men", "#eb6834")):
    axes[0].hist(eo.loc[eo["group"] == group, "TPR"], bins=np.linspace(0, 1, 29),
                 alpha=0.6, color=color, label=name)
axes[0].set(xlabel="TPR", ylabel="draws", title=f"equalized odds: TPR over {N_DRAWS} draws")
axes[0].legend()
women = eo[eo["group"] == "female"]
axes[1].hist(women["equalized-odds gap"], bins=np.linspace(0, 0.5, 26), color="#888888")
axes[1].axvline(women.loc[women["draw"] == SEED, "equalized-odds gap"].iloc[0], color="k",
                label=f"draw {SEED}")
axes[1].axvline(results["unmitigated"]["gaps"]["equalized odds"], color="k", ls="--",
                label="unmitigated")
axes[1].set(xlabel="equalized-odds gap", ylabel="draws", title="equalized odds: the gap")
axes[1].legend()
plt.show()

# Questions

1. On draw 42: what did equalized odds change for women, and what for men?
   Who paid for the nearly equal error rates?
2. Demographic parity: when 26 % of women and 55 % of men are sick, what does
   an equal selection rate mean for each group?
3. Fit the ThresholdOptimizer on the TEST part and evaluate it there. How much
   better does it look - and why is that number worthless?
4. There are 7 sick women in the test part. Compute the 95 % interval of
   their TPR with `wilson`. How much would you trust any TPR of women here?
5. After two hundred draws: does your answer to question 1 still hold? Which
   number would you report for a randomized rule - and how?

In [ ]:
# Space for questions 3 and 4 - e.g. wilson(5, 7) for the women's TPR.

---

*The numbers are computed by [`heart.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/bias-fairness/code/heart.py) of
the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/bias-fairness/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*